# LeetCode 2226 — Maximum Candies Allocated to K Children bằng Python

Link đề: https://leetcode.com/problems/maximum-candies-allocated-to-k-children/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def maximumCandies(self, candies: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển câu hỏi tối ưu "mỗi bé được **nhiều nhất** bao nhiêu kẹo" thành câu hỏi kiểm tra "mỗi bé được `x` kẹo **có khả thi** không": khả thi ⇔ `Σ ⌊c_i / x⌋ ≥ k`.
2. Nhận ra hàm khả thi **đơn điệu** (khả thi với `x` thì khả thi với mọi `x' < x`) ⇒ **tìm nhị phân theo đáp án**.
3. Viết tìm nhị phân "tìm giá trị lớn nhất thỏa điều kiện" **không lặp vô hạn**: dùng `mid = (lo + hi + 1) // 2` và bất biến rõ ràng.
4. Chặn trên đáp án chặt: `min(max(candies), sum(candies) // k)`; xử lý trường hợp đáp án `0`.
5. Kiểm thử đối chiếu quét tuyến tính trên dữ liệu ngẫu nhiên seed cố định, kể cả `k` rất lớn (tới `10¹²`).

## 2. Đề bài & ví dụ

Có các đống kẹo `candies[i]`. Bạn có thể **chia** mỗi đống thành bao nhiêu đống con tùy ý, nhưng **không được gộp** hai đống. Chia kẹo cho `k` bé sao cho **mỗi bé được cùng số kẹo** và **mỗi bé chỉ lấy từ một đống** (một đống con). Một số đống có thể không dùng.

Trả về số kẹo **lớn nhất** mỗi bé có thể nhận.

**Ví dụ 1:** `candies = [5,8,6], k = 3` → `5`: chia `8 → 5 + 3`, `6 → 5 + 1`; ba đống 5 cho ba bé.

**Ví dụ 2:** `candies = [2,5], k = 11` → `0`: chỉ có 7 kẹo cho 11 bé.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ candies[i] ≤ 10⁷`, `1 ≤ k ≤ 10¹²`.

In [1]:
import random
import time
from bisect import bisect_right
from typing import List

examples = [([5, 8, 6], 3, 5), ([2, 5], 11, 0)]
for c, k, exp in examples:
    print(f"candies={c}, k={k} -> kỳ vọng {exp}")

candies=[5, 8, 6], k=3 -> kỳ vọng 5
candies=[2, 5], k=11 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Một đống cho bao nhiêu phần `x` kẹo?

Đống `c` chia được tối đa `⌊c / x⌋` đống con kích thước `x` (phần dư bỏ đi). Tổng số bé được phục vụ với mức `x` là `Σ ⌊c_i / x⌋`.

In [2]:
def children_served(candies, x):
    return sum(c // x for c in candies)

for x in range(1, 9):
    print(f"x={x}: phục vụ được {children_served([5, 8, 6], x)} bé")

x=1: phục vụ được 19 bé
x=2: phục vụ được 9 bé
x=3: phục vụ được 5 bé
x=4: phục vụ được 4 bé
x=5: phục vụ được 3 bé
x=6: phục vụ được 2 bé
x=7: phục vụ được 1 bé
x=8: phục vụ được 1 bé


### 3.2 Hàm đơn điệu và tìm nhị phân theo đáp án

`f(x) = Σ ⌊c_i/x⌋` **không tăng** theo `x`. Vậy tập `{x : f(x) ≥ k}` có dạng `{1, 2, …, X}` (hoặc rỗng) — đáp án là `X`. Tìm `X` bằng tìm nhị phân trên `[0, hi]`.

In [3]:
vals = [children_served([5, 8, 6], x) for x in range(1, 9)]
print("f(x) =", vals, "| không tăng?", all(vals[i] >= vals[i + 1] for i in range(len(vals) - 1)))
print("khả thi với k=3:", [v >= 3 for v in vals])

f(x) = [19, 9, 5, 4, 3, 2, 1, 1] | không tăng? True
khả thi với k=3: [True, True, True, True, True, False, False, False]


### 3.3 Mẫu tìm nhị phân "lớn nhất thỏa"

Bất biến: `ok(lo)` luôn đúng (ta cho `lo = 0` — luôn "khả thi" theo nghĩa mỗi bé 0 kẹo), `ok(hi + 1)` luôn sai. Lặp khi `lo < hi`: `mid = (lo + hi + 1) // 2` (làm tròn **lên** để `mid > lo`, tránh lặp vô hạn); nếu `ok(mid)` thì `lo = mid`, ngược lại `hi = mid − 1`.

In [4]:
def largest_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid): lo = mid
        else: hi = mid - 1
    return lo

print(largest_true(0, 100, lambda x: x * x <= 50))

7


### 3.4 `bisect` với tham số `key` (Python 3.10+)

`bisect_right(range(lo, hi+1), 0, key=lambda x: −ok(x))`… — cách viết ngắn hơn cho tìm nhị phân trên một dãy đơn điệu "ảo" mà không tạo list (vì `range` hỗ trợ truy cập chỉ số).

In [5]:
r = range(1, 101)
idx = bisect_right(r, False, key=lambda x: not (x * x <= 50))   # số phần tử có "not ok" == False
print("x lớn nhất có x*x <= 50:", r[idx - 1] if idx else 0)

x lớn nhất có x*x <= 50: 7


### 3.5 Vì sao `⌊c/x⌋` không tăng theo `x`?

Nếu `x ≤ y` thì `c/x ≥ c/y` nên `⌊c/x⌋ ≥ ⌊c/y⌋` (phần nguyên bảo toàn thứ tự). Cộng các số không tăng vẫn được hàm không tăng. Ô dưới in `⌊8/x⌋` cho `x = 1..9` — các "bậc thang" chỉ đi xuống.

In [6]:
print([(x, 8 // x) for x in range(1, 10)])
print("divmod(8, 3) =", divmod(8, 3), "-> 2 đống con cỡ 3, dư 2 kẹo bỏ đi")

[(1, 8), (2, 4), (3, 2), (4, 2), (5, 1), (6, 1), (7, 1), (8, 1), (9, 0)]
divmod(8, 3) = (2, 2) -> 2 đống con cỡ 3, dư 2 kẹo bỏ đi


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `[5, 8, 6]`, `k = 3`. Thử mức `x`:

| x | ⌊5/x⌋ | ⌊8/x⌋ | ⌊6/x⌋ | tổng | ≥ 3? |
|---|---|---|---|---|---|
| 4 | 1 | 2 | 1 | 4 | ✓ |
| 5 | 1 | 1 | 1 | 3 | ✓ |
| 6 | 0 | 1 | 1 | 2 | ✗ |
| 7 | 0 | 1 | 0 | 1 | ✗ |

Mức lớn nhất khả thi là `5`. Chú ý `sum // k = 19 // 3 = 6` **không** đạt được vì không được gộp đống (đống 5 không thể "góp" cho đống khác).

Quan sát: ranh giới khả thi/không khả thi là **một điểm cắt** ⇒ tìm nhị phân.

Bảng khả thi đầy đủ cho cả hai ví dụ (`✓` = khả thi). Lưu ý ví dụ 2: ngay cả `x = 1` cũng không khả thi (chỉ có 7 kẹo cho 11 bé) ⇒ đáp án là `0`, nên tìm nhị phân phải cho phép `lo = 0`.

In [7]:
for c, k, _ in examples:
    marks = "".join("✓" if sum(v // x for v in c) >= k else "✗" for x in range(1, max(c) + 1))
    print(f"candies={c}, k={k}: x = 1..{max(c)} -> {marks}")

candies=[5, 8, 6], k=3: x = 1..8 -> ✓✓✓✓✓✗✗✗
candies=[2, 5], k=11: x = 1..5 -> ✗✗✗✗✗


## 5. Cách 1 — Brute Force / baseline: quét `x` từ lớn xuống nhỏ

**Ý tưởng:** thử `x = max(candies), max − 1, …, 1`; mức đầu tiên có `f(x) ≥ k` là đáp án; không có thì `0`.

In [8]:
def candies_scan(candies, k):
    for x in range(max(candies), 0, -1):
        if sum(c // x for c in candies) >= k:
            return x
    return 0

for c, k, exp in examples:
    got = candies_scan(c, k); print(got, "kỳ vọng", exp); assert got == exp

5 kỳ vọng 5
0 kỳ vọng 0


**Độ phức tạp:** tới `max(candies) = 10⁷` mức × `n = 10⁵` ⇒ `10¹²` — không thể. Space `O(1)`.

**Nút thắt:** ta thử từng mức một trong khi hàm khả thi đơn điệu — chỉ cần `log₂(10⁷) ≈ 24` lần thử.

In [9]:
rng = random.Random(0)
c = [rng.randint(1, 10**4) for _ in range(1000)]
for k in (10**3, 10**5, 10**7):
    t = time.perf_counter(); r = candies_scan(c, k)
    print(f"k={k}: x={r} ({time.perf_counter() - t:.3f}s)")

k=1000: x=3302 (0.141s)


k=100000: x=48 (0.208s)


k=10000000: x=0 (0.210s)


## 6. Ý tưởng thuật toán / bước đột phá

**Định lý đơn điệu:** nếu mỗi bé nhận được `x` kẹo thì cũng nhận được `x − 1` (mỗi đống con `x` bỏ bớt 1 kẹo). Hình thức: `⌊c/x⌋ ≤ ⌊c/(x−1)⌋`.

⇒ Đáp án `X = max{x ≥ 0 : f(x) ≥ k}` (quy ước `f(0) = ∞`, tức `x = 0` luôn khả thi).

**Chặn trên:** `X ≤ max(candies)` (một bé lấy từ một đống) và `X ≤ ⌊sum/k⌋` (tổng kẹo). Lấy `hi = min(max(candies), sum // k)`.

**Quy tắc "nếu… thì…" trong tìm nhị phân:** `mid = (lo + hi + 1) // 2`; **nếu** `f(mid) ≥ k` **thì** đáp án `≥ mid` ⇒ `lo = mid`; **ngược lại** đáp án `< mid` ⇒ `hi = mid − 1`.

**Invariant:** `f(lo) ≥ k` (hoặc `lo = 0`) và mọi `x > hi` đều không khả thi.

**Hai mẫu tìm nhị phân đối xứng.** Cùng một vị từ đơn điệu `ok(x)` ("đúng…đúng sai…sai"), ta có thể hỏi "x **lớn nhất** có `ok`" hoặc với vị từ ngược `bad(x) = not ok(x)` ("sai…sai đúng…đúng") hỏi "x **nhỏ nhất** có `bad`" — hai đáp án chênh nhau đúng 1. Ô dưới cài cả hai mẫu và kiểm chúng nhất quán; chọn mẫu theo cách đề hỏi giúp tránh lỗi làm tròn.

In [10]:
def smallest_true(lo, hi, pred):          # pred: sai...sai đúng...đúng; trả hi + 1 nếu không có
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid): hi = mid
        else: lo = mid + 1
    return lo

c, k = [5, 8, 6], 3
ok = lambda x: x == 0 or sum(v // x for v in c) >= k
big_ok = largest_true(0, 9, ok)
first_bad = smallest_true(0, 10, lambda x: not ok(x))
print("lớn nhất ok:", big_ok, "| nhỏ nhất không ok:", first_bad, "| chênh 1?", first_bad == big_ok + 1)

lớn nhất ok: 5 | nhỏ nhất không ok: 6 | chênh 1? True


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quét từ chặn trên chặt `min(max, sum // k)`

Vẫn tuyến tính nhưng bắt đầu từ chặn trên tốt hơn — với `k` lớn, chặn trên nhỏ nên quét rất ngắn; với `k` nhỏ vẫn có thể dài.

In [11]:
def candies_scan_tight(candies, k):
    hi = min(max(candies), sum(candies) // k)
    for x in range(hi, 0, -1):
        if sum(c // x for c in candies) >= k:
            return x
    return 0

for c, k, exp in examples:
    assert candies_scan_tight(c, k) == exp
print([candies_scan_tight(c, k) for c, k, _ in examples])

[5, 0]


**Độ phức tạp Cách 2:** `O(n · (hi − X + 1))` — tốt khi đáp án gần chặn trên, worst vẫn lớn.

### 7.2 Cách 3 — Tìm nhị phân theo đáp án (bản nộp chính)

In [12]:
def candies_bsearch(candies, k):
    lo, hi = 0, min(max(candies), sum(candies) // k)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(c // mid for c in candies) >= k:
            lo = mid
        else:
            hi = mid - 1
    return lo

for c, k, exp in examples:
    assert candies_bsearch(c, k) == exp
print([candies_bsearch(c, k) for c, k, _ in examples])

[5, 0]


**Độ phức tạp Cách 3:** `O(n · log(hi))` ≈ `10⁵ · 24`. Space `O(1)`.

### 7.3 Cách 4 — Tìm nhị phân + dừng sớm khi đếm đã đủ `k`

Trong hàm kiểm tra, cộng dồn và **dừng ngay** khi đạt `k` — có ích khi `k` nhỏ so với tổng.

In [13]:
def feasible(candies, x, k):
    total = 0
    for c in candies:
        total += c // x
        if total >= k:
            return True
    return False

def candies_bsearch_early(candies, k):
    lo, hi = 0, min(max(candies), sum(candies) // k)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if feasible(candies, mid, k): lo = mid
        else: hi = mid - 1
    return lo

for c, k, exp in examples:
    assert candies_bsearch_early(c, k) == exp
print([candies_bsearch_early(c, k) for c, k, _ in examples])

[5, 0]


### 7.4 Cách 5 — `bisect_right` với `key` trên `range`

Dãy `range(1, hi + 1)` ánh xạ qua `key(x) = (f(x) < k)` cho `False, …, False, True, …, True`. `bisect_right(…, False, key=…)` trả về số phần tử `False` = đáp án.

In [14]:
def candies_bisect(candies, k):
    hi = min(max(candies), sum(candies) // k)
    return bisect_right(range(1, hi + 1), False, key=lambda x: sum(c // x for c in candies) < k)

for c, k, exp in examples:
    assert candies_bisect(c, k) == exp
print([candies_bisect(c, k) for c, k, _ in examples])

[5, 0]


**Độ phức tạp Cách 5:** như Cách 3 (`bisect` gọi `key` `O(log hi)` lần).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 (`hi = min(8, 19 // 3) = 6`):

| lo | hi | mid | f(mid) | f ≥ 3? | cập nhật |
|---|---|---|---|---|---|
| 0 | 6 | 3 | 1+2+2 = 5 | ✓ | lo = 3 |
| 3 | 6 | 5 | 1+1+1 = 3 | ✓ | lo = 5 |
| 5 | 6 | 6 | 0+1+1 = 2 | ✗ | hi = 5 |

`lo = hi = 5` ⇒ đáp án `5`.

In [15]:
def trace_bsearch(candies, k):
    lo, hi = 0, min(max(candies), sum(candies) // k)
    print(f"bắt đầu lo={lo}, hi={hi}")
    while lo < hi:
        mid = (lo + hi + 1) // 2
        f = sum(c // mid for c in candies)
        if f >= k: lo = mid; act = "khả thi -> lo = mid"
        else: hi = mid - 1; act = "không -> hi = mid - 1"
        print(f"  mid={mid}: f(mid)={f} {'>=' if f >= k else '<'} {k} {act} | lo={lo}, hi={hi}")
    return lo

print("Đáp án:", trace_bsearch([5, 8, 6], 3))
print()
print("Đáp án:", trace_bsearch([2, 5], 11))

bắt đầu lo=0, hi=6
  mid=3: f(mid)=5 >= 3 khả thi -> lo = mid | lo=3, hi=6
  mid=5: f(mid)=3 >= 3 khả thi -> lo = mid | lo=5, hi=6
  mid=6: f(mid)=2 < 3 không -> hi = mid - 1 | lo=5, hi=5
Đáp án: 5

bắt đầu lo=0, hi=0
Đáp án: 0


In [16]:
print("Đáp án:", trace_bsearch([10**7, 3, 9_999_999], 4))

bắt đầu lo=0, hi=5000000
  mid=2500000: f(mid)=7 >= 4 khả thi -> lo = mid | lo=2500000, hi=5000000
  mid=3750000: f(mid)=4 >= 4 khả thi -> lo = mid | lo=3750000, hi=5000000
  mid=4375000: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4375000, hi=5000000
  mid=4687500: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4687500, hi=5000000
  mid=4843750: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4843750, hi=5000000
  mid=4921875: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4921875, hi=5000000
  mid=4960938: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4960938, hi=5000000
  mid=4980469: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4980469, hi=5000000
  mid=4990235: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4990235, hi=5000000
  mid=4995118: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4995118, hi=5000000
  mid=4997559: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4997559, hi=5000000
  mid=4998780: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4998780, hi=5000000
  mid=4999390: f(mid)=4 >= 4 khả thi -> lo = mid | lo=4999390, hi=5000000
  mid=4999695

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def maximumCandies(self, candies: List[int], k: int) -> int:
        lo, hi = 0, min(max(candies), sum(candies) // k)    # đáp án nằm trong [0, hi]
        while lo < hi:
            mid = (lo + hi + 1) // 2                       # làm tròn lên: tránh lặp vô hạn
            if sum(c // mid for c in candies) >= k:        # mỗi bé mid kẹo có khả thi?
                lo = mid
            else:
                hi = mid - 1
        return lo

for c, k, exp in examples:
    assert Solution().maximumCandies(c, k) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Khả thi ⇔ `f(x) ≥ k`.** Mỗi bé nhận một đống con kích thước đúng `x` (nhận nhiều hơn rồi bỏ bớt cũng tương đương). Đống `c_i` cho tối đa `⌊c_i/x⌋` đống con kích thước `x`, và các đống độc lập ⇒ tối đa `f(x)` bé được phục vụ. Ngược lại, nếu `f(x) ≥ k` thì chọn `k` trong số các đống con đó.

**Đơn điệu.** `x ≤ y ⇒ ⌊c/x⌋ ≥ ⌊c/y⌋` ⇒ `f` không tăng ⇒ tập khả thi là đoạn `[0, X]`.

**Tìm nhị phân.** Bất biến: `lo` khả thi (ban đầu `lo = 0` khả thi tầm thường), mọi `x > hi` không khả thi (ban đầu `hi` là chặn trên hợp lệ: nếu `x > max` thì `f(x) = 0 < k`; nếu `x > sum // k` thì `f(x) ≤ sum / x < k`). Mỗi bước giữ bất biến và `hi − lo` giảm (vì `mid > lo` nhờ làm tròn lên) ⇒ dừng với `lo = hi = X`.

In [18]:
# Kiểm chứng đơn điệu và chặn trên trên dữ liệu ngẫu nhiên
rng = random.Random(10)
for _ in range(300):
    c = [rng.randint(1, 50) for _ in range(rng.randint(1, 8))]; k = rng.randint(1, 60)
    f = [sum(x // m for x in c) for m in range(1, max(c) + 2)]
    assert all(f[i] >= f[i + 1] for i in range(len(f) - 1))
    X = candies_scan(c, k)
    assert X <= min(max(c), sum(c) // k)
print("f(x) không tăng và đáp án <= min(max, sum // k) trên 300 bộ")

f(x) không tăng và đáp án <= min(max, sum // k) trên 300 bộ


Kiểm chứng **bất biến của tìm nhị phân** ngay trong vòng lặp: ở mọi bước, `lo` khả thi và `hi + 1` không khả thi (dùng vét cạn để biết đáp án thật `X`, rồi kiểm `lo ≤ X ≤ hi`).

In [19]:
rng = random.Random(11)
for _ in range(300):
    c = [rng.randint(1, 40) for _ in range(rng.randint(1, 6))]; k = rng.randint(1, 50)
    X = candies_scan(c, k)
    lo, hi = 0, min(max(c), sum(c) // k)
    while lo < hi:
        assert lo <= X <= hi
        mid = (lo + hi + 1) // 2
        if sum(v // mid for v in c) >= k: lo = mid
        else: hi = mid - 1
    assert lo == X
print("Bất biến lo <= X <= hi đúng ở mọi bước (300 bộ)")

Bất biến lo <= X <= hi đúng ở mọi bước (300 bộ)


## 11. Phân tích Time / Space

Gọi `n = len(candies)`, `M = max(candies) ≤ 10⁷`.

| Cách | Time | Space |
|---|---|---|
| 1. Quét từ `M` | `O(n · M)` | `O(1)` |
| 2. Quét từ chặn chặt | `O(n · (hi − X))` | `O(1)` |
| 3. Tìm nhị phân | `O(n log M)` | `O(1)` |
| 4. + dừng sớm | `O(n log M)` (thường nhanh hơn) | `O(1)` |
| 5. `bisect` + `key` | `O(n log M)` | `O(1)` |

`k` tới `10¹²` và `sum` tới `10¹²` — vượt `int32` (Java/C++ dùng `long` cho tổng).

In [20]:
rng = random.Random(2226)
big = [rng.randint(1, 10**7) for _ in range(10**5)]
for k in (1, 10**5, 10**9, 10**12):
    row = []
    for name, fn in [("bsearch", candies_bsearch), ("early", candies_bsearch_early), ("bisect", candies_bisect)]:
        t = time.perf_counter(); r = fn(big, k); row.append(f"{name}={r} ({time.perf_counter() - t:.2f}s)")
    print(f"k={k:>14}: " + ", ".join(row))

k=             1: bsearch=9999933 (0.05s), early=9999933 (0.02s), bisect=9999933 (0.05s)


k=        100000: bsearch=3327547 (0.05s), early=3327547 (0.06s), bisect=3327547 (0.05s)


k=    1000000000: bsearch=498 (0.03s), early=498 (0.03s), bisect=498 (0.02s)
k= 1000000000000: bsearch=0 (0.00s), early=0 (0.00s), bisect=0 (0.00s)


## 12. Bộ test

- Ví dụ đề.
- Biên: `k = 1` (đáp án `max`); `k = sum` (đáp án 1); `k > sum` (0); một đống; mọi đống bằng nhau; `k = 10¹²`.
- Đối chiếu quét tuyến tính với các cách tìm nhị phân trên dữ liệu ngẫu nhiên nhỏ (seed cố định).

In [21]:
fns = {"scan": candies_scan, "scan_tight": candies_scan_tight, "bsearch": candies_bsearch,
       "early": candies_bsearch_early, "bisect": candies_bisect, "Solution": lambda c, k: Solution().maximumCandies(c, k)}
edge = [([7], 1, 7), ([7], 7, 1), ([7], 8, 0), ([4, 4, 4], 3, 4), ([4, 4, 4], 4, 2), ([1, 2, 3], 6, 1),
        ([10**7], 10**12, 0), ([10**7] * 3, 10**7, 2)]
for c, k, exp in examples + edge:
    for name, fn in fns.items():
        if name in ("scan",) and max(c) > 10**5: continue
        assert fn(c, k) == exp, (name, c, k, fn(c, k))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [22]:
rng = random.Random(42)
for _ in range(400):
    c = [rng.randint(1, rng.choice([5, 50, 500])) for _ in range(rng.randint(1, 10))]
    k = rng.randint(1, sum(c) + 3)
    want = candies_scan(c, k)
    for name, fn in fns.items():
        assert fn(c, k) == want, (name, c, k)
print("400 test ngẫu nhiên: mọi cách khớp quét tuyến tính")

400 test ngẫu nhiên: mọi cách khớp quét tuyến tính


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Trả `sum(candies) // k` (quên "không được gộp đống")

In [23]:
def wrong_average(candies, k):
    return sum(candies) // k                              # SAI

print("sai:", wrong_average([5, 8, 6], 3), "| đúng:", 5)

sai: 6 | đúng: 5


In [24]:
print("sửa:", Solution().maximumCandies([5, 8, 6], 3))

sửa: 5


### Lỗi 2 — `mid = (lo + hi) // 2` với cập nhật `lo = mid` (lặp vô hạn)

Khi `hi = lo + 1` và `mid = lo` khả thi, `lo` không đổi ⇒ vòng lặp không bao giờ dừng. Ta demo với "cầu chì" giới hạn số vòng.

In [25]:
def wrong_mid_floor(candies, k, cap=100):
    lo, hi = 0, min(max(candies), sum(candies) // k); steps = 0
    while lo < hi:
        steps += 1
        if steps > cap: return f"lặp vô hạn (lo={lo}, hi={hi})"
        mid = (lo + hi) // 2                              # SAI: làm tròn xuống
        if mid == 0 or sum(c // mid for c in candies) >= k: lo = mid
        else: hi = mid - 1
    return lo

print("sai:", wrong_mid_floor([5, 8, 6], 3), "| đúng:", 5)

sai: lặp vô hạn (lo=5, hi=6) | đúng: 5


In [26]:
print("sửa (làm tròn lên):", candies_bsearch([5, 8, 6], 3))

sửa (làm tròn lên): 5


### Lỗi 3 — Bắt đầu tìm từ `lo = 1` rồi chia cho `mid = 0` hoặc trả 1 khi đáp án là 0

Nếu `lo = 1` mà thực ra không khả thi (ví dụ 2), bản sai vẫn trả `1`.

In [27]:
def wrong_lo_one(candies, k):
    lo, hi = 1, max(candies)                              # SAI: giả định 1 luôn khả thi
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(c // mid for c in candies) >= k: lo = mid
        else: hi = mid - 1
    return lo

print("sai:", wrong_lo_one([2, 5], 11), "| đúng:", 0)

sai: 1 | đúng: 0


In [28]:
print("sửa (lo = 0):", Solution().maximumCandies([2, 5], 11))

sửa (lo = 0): 0


### Lỗi 4 — So sánh `> k` thay vì `>= k`

In [29]:
def wrong_strict(candies, k):
    lo, hi = 0, max(candies)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(c // mid for c in candies) > k: lo = mid      # SAI: đủ đúng k bé là được
        else: hi = mid - 1
    return lo

print("sai:", wrong_strict([5, 8, 6], 3), "| đúng:", 5)

sai: 4 | đúng: 5


In [30]:
print("sửa:", candies_bsearch([5, 8, 6], 3))

sửa: 5


### Lỗi 5 — Giới hạn trên là `min(candies)` (nghĩ mỗi bé không thể nhận hơn đống nhỏ nhất)

Các đống nhỏ có thể **bỏ không dùng**.

In [31]:
def wrong_cap_min(candies, k):
    lo, hi = 0, min(candies)                              # SAI
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(c // mid for c in candies) >= k: lo = mid
        else: hi = mid - 1
    return lo

print("sai:", wrong_cap_min([1, 100, 100], 2), "| đúng:", Solution().maximumCandies([1, 100, 100], 2))

sai: 1 | đúng: 100


In [32]:
print("sửa (hi = min(max, sum // k)):", candies_bsearch([1, 100, 100], 2))

sửa (hi = min(max, sum // k)): 100


### Lỗi 6 — (Khi dịch sang Java/C++) cộng dồn `Σ ⌊c/x⌋` bằng `int`

Với `x = 1`, tổng có thể tới `10⁵ · 10⁷ = 10¹²` — tràn `int32`, so sánh với `k` sai. Python không gặp, nhưng ta mô phỏng phép cộng 32-bit để thấy.

In [33]:
def add32(a, b):
    s = (a + b) & 0xFFFFFFFF
    return s - (1 << 32) if s >= 1 << 31 else s

c = [10**7] * 10**5; k = 10**12
total32 = 0
for v in c: total32 = add32(total32, v // 1)
print("tổng đúng:", sum(c), "| tổng int32:", total32, "| so sánh >= k: đúng", sum(c) >= k, "vs int32", total32 >= k)

tổng đúng: 1000000000000 | tổng int32: -727379968 | so sánh >= k: đúng True vs int32 False


In [34]:
print("sửa (64-bit / Python int):", Solution().maximumCandies(c, k))

sửa (64-bit / Python int): 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tối đa giá trị `x` sao cho `điều kiện(x)`"** + điều kiện **đơn điệu** ⇒ tìm nhị phân theo đáp án.
- Hàm kiểm tra thường là một **tổng** `Σ ⌊c_i/x⌋` hoặc `Σ ⌈c_i/x⌉` — `O(n)` mỗi lần.
- Mẫu "lớn nhất thỏa": `mid = (lo + hi + 1) // 2`, `lo = mid` / `hi = mid − 1`. Mẫu "nhỏ nhất thỏa": `mid = (lo + hi) // 2`, `hi = mid` / `lo = mid + 1`.
- Luôn xét đáp án biên (`0`) và chặn trên chặt.
- Ràng buộc `10¹²` ⇒ dùng số nguyên 64-bit ở ngôn ngữ khác.

## 15. Biến thể của bài

1. **LeetCode 1760 — Minimum Limit of Balls in a Bag:** chia túi tối đa `maxOperations` lần để **tối thiểu** túi lớn nhất (bài tập 5) — "nhỏ nhất thỏa".
2. **LeetCode 875 — Koko Eating Bananas:** tốc độ nhỏ nhất để ăn xong trong `h` giờ — `Σ ⌈c/x⌉ ≤ h`.
3. **LeetCode 1283 — Find the Smallest Divisor Given a Threshold.**
4. **LeetCode 410 — Split Array Largest Sum (Hard):** tìm nhị phân + tham lam kiểm tra.
5. **LeetCode 2064 — Minimized Maximum of Products Distributed to Any Store.**

Minh họa biến thể 2 (Koko, tự dựng, kiểm bằng quét):

In [35]:
def koko(piles, h):
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((p + mid - 1) // mid for p in piles) <= h: hi = mid
        else: lo = mid + 1
    return lo

def koko_scan(piles, h):
    return next(x for x in range(1, max(piles) + 1) if sum((p + x - 1) // x for p in piles) <= h)

rng = random.Random(875)
for _ in range(300):
    piles = [rng.randint(1, 40) for _ in range(rng.randint(1, 6))]; h = rng.randint(len(piles), len(piles) + 30)
    assert koko(piles, h) == koko_scan(piles, h)
print("Koko (nhỏ nhất thỏa) khớp quét; [3,6,7,11], h=8 ->", koko([3, 6, 7, 11], 8))

Koko (nhỏ nhất thỏa) khớp quét; [3,6,7,11], h=8 -> 4


Minh họa biến thể 3 — **LeetCode 1283 (Smallest Divisor Given a Threshold)**: tìm số chia `d` nhỏ nhất sao cho `Σ ⌈a_i / d⌉ ≤ threshold`. Lại là "nhỏ nhất thỏa" với vị từ đơn điệu (tăng `d` thì tổng giảm).

In [36]:
def smallest_divisor(nums, threshold):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((a + mid - 1) // mid for a in nums) <= threshold: hi = mid
        else: lo = mid + 1
    return lo

rng = random.Random(1283)
for _ in range(300):
    nums = [rng.randint(1, 50) for _ in range(rng.randint(1, 6))]; th = rng.randint(len(nums), len(nums) + 40)
    brute = next(d for d in range(1, max(nums) + 1) if sum((a + d - 1) // d for a in nums) <= th)
    assert smallest_divisor(nums, th) == brute
print("LeetCode 1283 khớp quét; [1,2,5,9], 6 ->", smallest_divisor([1, 2, 5, 9], 6))

LeetCode 1283 khớp quét; [1,2,5,9], 6 -> 5


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét từ max | thử từng mức | `O(n·M)` | `O(1)` | kiểm tra |
| 2. Quét từ chặn chặt | bắt đầu `min(max, sum//k)` | `O(n·(hi−X))` | `O(1)` | `k` rất lớn |
| 3. Tìm nhị phân | điều kiện đơn điệu | `O(n log M)` | `O(1)` | **nộp bài** |
| 4. + dừng sớm | cắt ngắn hàm kiểm tra | `O(n log M)` | `O(1)` | `k` nhỏ |
| 5. `bisect` + `key` | thư viện chuẩn | `O(n log M)` | `O(1)` | viết gọn |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `maximumCandies([4, 7, 5], 4)`.
2. **Tính tay:** trace tìm nhị phân cho `[9, 10, 1, 2, 10, 9, 9, 10, 2, 2]`, `k = 3`.
3. **Điền TODO:** `upper_bound(candies, k)` trả về chặn trên chặt.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `min_max_ball(nums, maxOperations)` (LeetCode 1760): mỗi thao tác chia một túi thành hai túi khác rỗng; sau tối đa `maxOperations` thao tác, tối thiểu kích thước túi lớn nhất. Kiểm bằng quét tuyến tính.

In [37]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def upper_bound(candies, k):
    # TODO
    pass

In [38]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_maximum_candies(candies, k):
    # TODO
    pass

In [39]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def min_max_ball(nums, maxOperations):
    # TODO: với ngưỡng x, túi c cần (c - 1) // x thao tác; tìm x nhỏ nhất có tổng <= maxOperations
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `x = 4`: `1 + 1 + 1 = 3 < 4`. (2) `x = 3`: `1 + 2 + 1 = 4 ≥ 4`. (3) Đáp án `3`.

**Bài 2.** (1) `hi = min(10, 64 // 3 = 21) = 10`. (2) `mid = 5`: `f = 1+2+0+0+2+1+1+2+0+0 = 9 ≥ 3`. (3) Tiếp tục tới khi `f(mid) < 3` — đáp án `10` (ba đống 10).

**Bài 3.** (1) `max(candies)`. (2) `sum(candies) // k`. (3) Lấy `min`.

**Bài 4.** (1) `lo = 0`. (2) `mid = (lo + hi + 1) // 2`. (3) `f(mid) ≥ k` ⇒ `lo = mid`.

**Bài 5.** (1) Đây là bài "nhỏ nhất thỏa". (2) Túi `c` với ngưỡng `x` cần `⌈c/x⌉ − 1 = (c − 1) // x` lần chia. (3) `lo = 1`, `hi = max(nums)`, `mid = (lo + hi) // 2`.

## 19. Đáp án tham khảo

In [40]:
# Bài 1, 2
assert Solution().maximumCandies([4, 7, 5], 4) == 3
assert Solution().maximumCandies([9, 10, 1, 2, 10, 9, 9, 10, 2, 2], 3) == 10
print("Bài 1: 3 | Bài 2: 10")
trace_bsearch([9, 10, 1, 2, 10, 9, 9, 10, 2, 2], 3)

Bài 1: 3 | Bài 2: 10
bắt đầu lo=0, hi=10
  mid=5: f(mid)=9 >= 3 khả thi -> lo = mid | lo=5, hi=10
  mid=8: f(mid)=6 >= 3 khả thi -> lo = mid | lo=8, hi=10
  mid=9: f(mid)=6 >= 3 khả thi -> lo = mid | lo=9, hi=10
  mid=10: f(mid)=3 >= 3 khả thi -> lo = mid | lo=10, hi=10


10

In [41]:
# Bài 3, 4
def upper_bound_ans(candies, k):
    return min(max(candies), sum(candies) // k)

my_maximum_candies_ans = candies_bsearch
rng = random.Random(3)
for _ in range(300):
    c = [rng.randint(1, 60) for _ in range(rng.randint(1, 8))]; k = rng.randint(1, 80)
    X = candies_scan(c, k)
    assert X <= upper_bound_ans(c, k) and my_maximum_candies_ans(c, k) == X
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [42]:
# Bài 5
def min_max_ball_ans(nums, maxOperations):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((c - 1) // mid for c in nums) <= maxOperations: hi = mid
        else: lo = mid + 1
    return lo

def min_max_ball_scan(nums, maxOperations):
    return next(x for x in range(1, max(nums) + 1) if sum((c - 1) // x for c in nums) <= maxOperations)

rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(1, 50) for _ in range(rng.randint(1, 6))]; ops = rng.randint(0, 20)
    assert min_max_ball_ans(nums, ops) == min_max_ball_scan(nums, ops)
print("Bài 5: OK; [9], 2 ->", min_max_ball_ans([9], 2), "| [2,4,8,2], 4 ->", min_max_ball_ans([2, 4, 8, 2], 4))

Bài 5: OK; [9], 2 -> 3 | [2,4,8,2], 4 -> 2


**Giải thích thêm cho Bài 5.** Muốn mọi túi có kích thước `≤ x`, một túi `c` phải được chia thành ít nhất `⌈c/x⌉` túi, tức cần `⌈c/x⌉ − 1 = (c − 1) // x` lần chia (mỗi lần chia tăng số túi thêm 1). Tổng số lần chia giảm khi `x` tăng ⇒ "nhỏ nhất `x` có tổng ≤ `maxOperations`". Bảng dưới in số lần chia cần theo `x` cho `[2, 4, 8, 2]`.

In [43]:
nums = [2, 4, 8, 2]
for x in range(1, 9):
    need = sum((c - 1) // x for c in nums)
    print(f"x={x}: cần {need} lần chia {'(<= 4: ok)' if need <= 4 else ''}")

x=1: cần 12 lần chia 
x=2: cần 4 lần chia (<= 4: ok)
x=3: cần 3 lần chia (<= 4: ok)
x=4: cần 1 lần chia (<= 4: ok)
x=5: cần 1 lần chia (<= 4: ok)
x=6: cần 1 lần chia (<= 4: ok)
x=7: cần 1 lần chia (<= 4: ok)
x=8: cần 0 lần chia (<= 4: ok)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lớn nhất thỏa":**

```python
lo, hi = 0, upper                 # lo luôn thỏa, mọi x > hi đều không thỏa
while lo < hi:
    mid = (lo + hi + 1) // 2      # làm tròn LÊN
    if ok(mid): lo = mid
    else: hi = mid - 1
return lo
```

In [44]:
def template(candies, k):
    lo, hi = 0, min(max(candies), sum(candies) // k)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(c // mid for c in candies) >= k: lo = mid
        else: hi = mid - 1
    return lo
print([template(c, k) for c, k, _ in examples])

[5, 0]


**Checklist:**

- [ ] Hàm kiểm tra `Σ ⌊c_i/x⌋ ≥ k` (`≥`, không `>`).
- [ ] `lo = 0` (đáp án có thể 0), `hi = min(max, sum // k)`.
- [ ] `mid = (lo + hi + 1) // 2` với `lo = mid` (tránh lặp vô hạn, tránh chia 0).
- [ ] Không trả `sum // k` (không được gộp đống).
- [ ] Tổng/`k` tới `10¹²` (64-bit).
- [ ] Đã thử `k = 1`, `k = sum`, `k > sum`.